# Step 7: Evaluasi Model

Menggunakan **7 indikator evaluasi** sesuai C23 paper (Tabel 4):

| Indikator | Definisi |
|-----------|----------|
| **MS** | Most Suitable: suit DAN kategori tepat |
| **SCA** | Same Category Acceptable: kategori sama, suit beda tapi wajar |
| **SCU** | Same Category Unacceptable: kategori sama, suit tidak ideal |
| **SSE** | Same Suit Excl. MS: suit sama, kategori beda |
| **O** | Others: sama sekali tidak cocok |
| **SC** | Same Category = MS+SCA+SCU ← **METRIK UTAMA** (target ≥ 0.773) |
| **SS** | Same Suit = MS+SSE |

**Prasyarat:** Jalankan `05_dataset.ipynb` dan `06_training.ipynb` terlebih dahulu.

In [ ]:
import sys
import os

ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
sys.path.insert(0, os.path.join(ROOT, "src"))

from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
from IPython.display import Image, display

DATA_PROCESSED = Path(ROOT) / "data" / "processed"
RESULTS        = Path(ROOT) / "results"

SPLIT_PATH  = RESULTS / "metrics" / "dataset_split.pkl"
MLP_PATH    = RESULTS / "metrics" / "mlp" / "model.keras"
LSTM_PATH   = RESULTS / "metrics" / "lstm" / "model.keras"
FIGURES_DIR = RESULTS / "figures"
METRICS_DIR = RESULTS / "metrics"

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)

# Override path di modul evaluate agar mengarah ke direktori proyek
import evaluate as ev_module
ev_module.FIGURES_DIR = FIGURES_DIR
ev_module.METRICS_DIR = METRICS_DIR

print(f"Root proyek : {ROOT}")
print("Setup selesai.")

## 7.1 Load Model dan Data Test

In [ ]:
from model import load_model
from features import FEATURE_COLS
from model import prepare_features
from sklearn.model_selection import train_test_split

# Load models
model_mlp = load_model(model_type="mlp", path=MLP_PATH)
model_lstm = load_model(model_type="lstm", path=LSTM_PATH)
print(f"Model MLP dimuat dari: {MLP_PATH}")
print(f"Model LSTM dimuat dari: {LSTM_PATH}")

# Load split
if SPLIT_PATH.exists():
    split_data   = joblib.load(SPLIT_PATH)
    X_test       = split_data["X_test"]
    y_suit_test  = split_data["y_suit_test"]
    y_cat_test   = split_data["y_cat_test"]
    X_train      = split_data["X_train"]
    y_suit_train = split_data["y_suit_train"]
    y_cat_train  = split_data["y_cat_train"]
    print(f"Split data dimuat: test={X_test.shape[0]} sampel")
else:
    DATASET_CSV = DATA_PROCESSED / "bridge_dataset.csv"
    df = pd.read_csv(DATASET_CSV).dropna(subset=["best_contract_strain", "best_contract_category"])
    available_feat = [c for c in FEATURE_COLS if c in df.columns]
    X      = prepare_features(df, available_feat)
    y_suit = df["best_contract_strain"]
    y_cat  = df["best_contract_category"]
    X_train, X_test, y_suit_train, y_suit_test = train_test_split(
        X, y_suit, test_size=0.2, stratify=y_suit, random_state=42
    )
    _, _, y_cat_train, y_cat_test = train_test_split(
        X, y_cat, test_size=0.2, stratify=y_suit, random_state=42
    )
    print("Split direkonstruksi dari dataset CSV")

## 7.2 Evaluasi Model MLP

In [ ]:
from evaluate import evaluate

print("=== Evaluasi Model MLP ===")
metrics_mlp = evaluate(
    model=model_mlp,
    X_test=X_test,
    y_suit_test=y_suit_test,
    y_cat_test=y_cat_test,
    model_name="TwoStageMLP",
    save_figures=True,
)

## 7.3 Evaluasi Model LSTM

In [ ]:
print("\n=== Evaluasi Model LSTM ===")
metrics_lstm = evaluate(
    model=model_lstm,
    X_test=X_test,
    y_suit_test=y_suit_test,
    y_cat_test=y_cat_test,
    model_name="TwoStageLSTM",
    save_figures=True,
)

## 7.4 Perbandingan Metrik

In [ ]:
def print_summary(model_name, metrics):
    print(f"\n=== RINGKASAN METRIK {model_name} ===")
    print(f"SC (Same Category) — METRIK UTAMA : {metrics['sc_accuracy']:.4f}")
    print(f"SS (Same Suit)                    : {metrics['ss_accuracy']:.4f}")
    print(f"MS (Most Suitable / exact match)  : {metrics['ms_accuracy']:.4f}")
    print(f"Stage 1 Suit F1 (weighted)        : {metrics['suit_f1']:.4f}")
    print(f"Stage 2 Category F1 (weighted)    : {metrics['category_f1']:.4f}")

print_summary("MLP", metrics_mlp)
print_summary("LSTM", metrics_lstm)

paper_sc = 0.773
print(f"\n=== PERBANDINGAN DENGAN PAPER ===")
print(f"Paper SC target: {paper_sc}")
print(f"MLP SC: {metrics_mlp['sc_accuracy']:.4f} (diff: {metrics_mlp['sc_accuracy']-paper_sc:+.4f})")
print(f"LSTM SC: {metrics_lstm['sc_accuracy']:.4f} (diff: {metrics_lstm['sc_accuracy']-paper_sc:+.4f})")

best_model = "MLP" if metrics_mlp['sc_accuracy'] >= metrics_lstm['sc_accuracy'] else "LSTM"
print(f"\nModel terbaik: {best_model}")

## 7.5 Bar Chart Perbandingan 7 Indikator

In [ ]:
from evaluate import plot_indicator_bar

print("=== MLP Indicator Chart ===")
plot_indicator_bar(metrics_mlp["indicators"], model_name="TwoStageMLP")

print("\n=== LSTM Indicator Chart ===")
plot_indicator_bar(metrics_lstm["indicators"], model_name="TwoStageLSTM")

## 7.6 Confusion Matrix

In [ ]:
print("=== Confusion Matrix MLP ===")
cm_suit_path_mlp = FIGURES_DIR / "confusion_matrix_TwoStageMLP_suit.png"
cm_cat_path_mlp  = FIGURES_DIR / "confusion_matrix_TwoStageMLP_category.png"

if cm_suit_path_mlp.exists():
    print("Stage 1: Suit")
    display(Image(filename=str(cm_suit_path_mlp), width=500))
if cm_cat_path_mlp.exists():
    print("\nStage 2: Category")
    display(Image(filename=str(cm_cat_path_mlp), width=500))

In [ ]:
print("\n=== Confusion Matrix LSTM ===")
cm_suit_path_lstm = FIGURES_DIR / "confusion_matrix_TwoStageLSTM_suit.png"
cm_cat_path_lstm  = FIGURES_DIR / "confusion_matrix_TwoStageLSTM_category.png"

if cm_suit_path_lstm.exists():
    print("Stage 1: Suit")
    display(Image(filename=str(cm_suit_path_lstm), width=500))
if cm_cat_path_lstm.exists():
    print("\nStage 2: Category")
    display(Image(filename=str(cm_cat_path_lstm), width=500))

## 7.7 Feature Importance

In [ ]:
print("=== MLP Feature Importance ===")
fi_path_mlp = FIGURES_DIR / "feature_importance_TwoStageMLP.png"
if fi_path_mlp.exists():
    display(Image(filename=str(fi_path_mlp), width=800))

imp_suit_mlp, imp_cat_mlp = model_mlp.feature_importance()
print("\nTop 10 Fitur — Stage 1 (Suit):")
print(imp_suit_mlp.head(10).to_string())
print("\nTop 10 Fitur — Stage 2 (Category):")
print(imp_cat_mlp.head(10).to_string())

In [ ]:
print("\n=== LSTM Feature Importance ===")
fi_path_lstm = FIGURES_DIR / "feature_importance_TwoStageLSTM.png"
if fi_path_lstm.exists():
    display(Image(filename=str(fi_path_lstm), width=800))

imp_suit_lstm, imp_cat_lstm = model_lstm.feature_importance()
print("\nTop 10 Fitur — Stage 1 (Suit):")
print(imp_suit_lstm.head(10).to_string())
print("\nTop 10 Fitur — Stage 2 (Category):")
print(imp_cat_lstm.head(10).to_string())

## 7.8 Simpan Ringkasan Metrik

In [ ]:
def save_summary(model_name, metrics, filename):
    summary_text = f"""=== RINGKASAN EVALUASI MODEL 2-STAGE {model_name} ===
Test set     : {X_test.shape[0]} sampel
Fitur        : {X_test.shape[1]}

=== 7 INDIKATOR (C23 Tabel 4) ===
{metrics['indicators'].to_string()}

=== PERBANDINGAN DENGAN C23 PAPER ===
SC (metrik utama): {metrics['sc_accuracy']:.4f} (paper: 0.773)
SS              : {metrics['ss_accuracy']:.4f}
MS              : {metrics['ms_accuracy']:.4f}

=== F1 PER STAGE ===
Stage 1 (Suit)     F1 weighted: {metrics['suit_f1']:.4f}
Stage 2 (Category) F1 weighted: {metrics['category_f1']:.4f}
"""
    summary_path = METRICS_DIR / filename
    summary_path.write_text(summary_text, encoding="utf-8")
    print(f"Ringkasan {model_name} disimpan ke: {summary_path}")
    return summary_text

summary_mlp = save_summary("MLP", metrics_mlp, "evaluation_summary_mlp.txt")
summary_lstm = save_summary("LSTM", metrics_lstm, "evaluation_summary_lstm.txt")

# Save comparison summary
comparison_text = f"""=== PERBANDINGAN MODEL MLP VS LSTM ===
Test set: {X_test.shape[0]} sampel

| Metrik           | MLP    | LSTM   | Lebih baik |
|------------------|--------|--------|------------|
| SC (Same Category) | {metrics_mlp['sc_accuracy']:.4f} | {metrics_lstm['sc_accuracy']:.4f} | {'MLP' if metrics_mlp['sc_accuracy'] > metrics_lstm['sc_accuracy'] else 'LSTM'} |
| SS (Same Suit)    | {metrics_mlp['ss_accuracy']:.4f} | {metrics_lstm['ss_accuracy']:.4f} | {'MLP' if metrics_mlp['ss_accuracy'] > metrics_lstm['ss_accuracy'] else 'LSTM'} |
| MS (Most Suitable) | {metrics_mlp['ms_accuracy']:.4f} | {metrics_lstm['ms_accuracy']:.4f} | {'MLP' if metrics_mlp['ms_accuracy'] > metrics_lstm['ms_accuracy'] else 'LSTM'} |
| Suit F1           | {metrics_mlp['suit_f1']:.4f} | {metrics_lstm['suit_f1']:.4f} | {'MLP' if metrics_mlp['suit_f1'] > metrics_lstm['suit_f1'] else 'LSTM'} |
| Category F1       | {metrics_mlp['category_f1']:.4f} | {metrics_lstm['category_f1']:.4f} | {'MLP' if metrics_mlp['category_f1'] > metrics_lstm['category_f1'] else 'LSTM'} |

=== KESIMPULAN ===
Model dengan performa terbaik (SC tertinggi): {'MLP' if metrics_mlp['sc_accuracy'] > metrics_lstm['sc_accuracy'] else 'LSTM'}
"""
comparison_path = METRICS_DIR / "evaluation_comparison.txt"
comparison_path.write_text(comparison_text, encoding="utf-8")
print(f"\nPerbandingan disimpan ke: {comparison_path}")
print("\n" + comparison_text)

---
## Output

File yang dihasilkan:
- `results/figures/confusion_matrix_TwoStageMLP_*.png`
- `results/figures/confusion_matrix_TwoStageLSTM_*.png`
- `results/figures/feature_importance_TwoStageMLP.png`
- `results/figures/feature_importance_TwoStageLSTM.png`
- `results/figures/indicators_TwoStageMLP.png`
- `results/figures/indicators_TwoStageLSTM.png`
- `results/metrics/evaluation_summary_mlp.txt`
- `results/metrics/evaluation_summary_lstm.txt`
- `results/metrics/evaluation_comparison.txt`

**Langkah berikutnya:** Buka `08_analisis.ipynb` untuk analisis mendalam hasil.